<a href="https://colab.research.google.com/github/sheikh-saadullah/python_task_programs/blob/main/python_task.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
print("Python_Class_Task")

Python_Class_Task


Task 01 | Nested Transaction Analyzer
- Concepts: Data structures, sorting, comprehensions
- Problem: Given a list of transaction dictionaries with keys id, customer, category, amount, and status, produce a report containing each customer’s successful spending, top category, and rank by spending. Ignore failed transactions and reject duplicate transaction IDs.
- Requirements: Return a list sorted by spending descending, then customer name ascending. Customers with no successful transactions must appear with total 0. Do not use pandas.
- Sample / expected behavior: Input: [{id:1,customer:"Ali",category:"Food",amount:500,status:"success"},{id:2,customer:"Ali",category:"Travel",amount:200,status:"failed"},{id:3,customer:"Sara",category:"Food",amount:700,status:"success"}]. Output totals: Sara=700, Ali=500.


In [ ]:
def build_report(transactions):
    """Return a spending report: one row per customer, ranked by spending."""
    seen_ids = set()          # every transaction id we've processed
    totals = {}               # customer -> total successful spending
    category_totals = {}      # customer -> {category -> successful spending}

    # ---- Pass 1: validate and accumulate ----
    for t in transactions:
        tid = t["id"]
        if tid in seen_ids:
            raise ValueError(f"Duplicate transaction id: {tid}")
        seen_ids.add(tid)

        customer = t["customer"]
        # Register the customer even if this transaction failed,
        # so customers with no successes still appear with total 0.
        totals.setdefault(customer, 0)
        category_totals.setdefault(customer, {})

        if t["status"] != "success":
            continue  # failed transactions never count

        amount = t["amount"]
        category = t["category"]
        totals[customer] += amount
        cats = category_totals[customer]
        cats[category] = cats.get(category, 0) + amount

    # ---- Pass 2: sort customers ----
    ordered = sorted(totals.items(), key=lambda item: (-item[1], item[0]))

    # ---- Pass 3: build the report with ranks ----
    report = []
    rank = 0
    previous_total = None
    for position, (customer, total) in enumerate(ordered, start=1):
        if total != previous_total:      # new total -> new rank
            rank = position
            previous_total = total

        cats = category_totals[customer]
        if cats:
            top_category = min(cats.items(), key=lambda kv: (-kv[1], kv[0]))[0]
        else:
            top_category = None          # no successful transactions

        report.append({
            "customer": customer,
            "total": total,
            "top_category": top_category,
            "rank": rank,
        })
    return report

In [ ]:
transactions = [
    {"id": 1, "customer": "Ali",  "category": "Food",   "amount": 500, "status": "success"},
    {"id": 2, "customer": "Ali",  "category": "Travel", "amount": 200, "status": "failed"},
    {"id": 3, "customer": "Sara", "category": "Food",   "amount": 700, "status": "success"},
]

report = build_report(transactions)   # call the function, store the result
print(report)

[{'customer': 'Sara', 'total': 700, 'top_category': 'Food', 'rank': 1}, {'customer': 'Ali', 'total': 500, 'top_category': 'Food', 'rank': 2}]


Task 02 | Longest Valid Window
- Concepts: Sliding window, dictionaries
Problem: Write longest_unique_window(text) to find the longest contiguous substring with no repeated characters. Return the substring, start index, and length.
- Requirements: Target O(n) time. On equal-length windows, select the earliest. Treat spaces and punctuation as characters.
- Sample / expected behavior: Input: "abcabcbb" -> ("abc", 0, 3); "" -> ("", 0, 0).


In [ ]:
def longest_unique_window(text):
    """Return (substring, start_index, length) of the longest
    substring with no repeated characters (earliest on ties)."""
    last_seen = {}      # char -> most recent index where it appeared
    left = 0            # start of the current unique window
    best_start = 0      # start of the best window found so far
    best_len = 0        # length of the best window found so far

    for right, ch in enumerate(text):
        # If ch is already inside the current window, shrink from the left.
        if ch in last_seen and last_seen[ch] >= left:
            left = last_seen[ch] + 1

        last_seen[ch] = right               # remember latest position of ch

        window_len = right - left + 1
        if window_len > best_len:           # strict > keeps the earliest on ties
            best_len = window_len
            best_start = left

    return (text[best_start:best_start + best_len], best_start, best_len)

In [ ]:
print(longest_unique_window("abcabcbb"))   # ('abc', 0, 3)
print(longest_unique_window(""))           # ('', 0, 0)

('abc', 0, 3)
('', 0, 0)


Task 03 | Recursive JSON Path Explorer
- Concepts: Recursion, nested lists/dictionaries
- Problem: Implement find_paths(data, target) that recursively searches nested dictionaries and lists and returns all paths whose value equals target.
- Requirements: Use dot notation for dictionary keys and [index] for list positions. Preserve traversal order; do not treat bool True as numeric 1.
- Sample / expected behavior: Input: {"a":[{"x":5},5],"b":{"y":5}}, target=5 -> ["a[0].x", "a[1]", "b.y"].


In [ ]:
def values_match(value, target):
    """Equality that never treats bool as a number."""
    if isinstance(value, bool) or isinstance(target, bool):
        # If either side is a bool, BOTH must be bools and equal.
        return isinstance(value, bool) and isinstance(target, bool) and value == target
    return value == target


def find_paths(data, target):
    """Return all paths (dot/[index] notation) whose value equals target."""
    results = []

    def walk(node, path):
        if isinstance(node, dict):
            for key, child in node.items():
                child_path = str(key) if path == "" else f"{path}.{key}"
                walk(child, child_path)
        elif isinstance(node, list):
            for index, child in enumerate(node):
                walk(child, f"{path}[{index}]")
        elif values_match(node, target):
            results.append(path)

    walk(data, "")
    return results

In [ ]:
data = {"a": [{"x": 5}, 5], "b": {"y": 5}}
print(find_paths(data, 5))

['a[0].x', 'a[1]', 'b.y']


Task 04 | Dependency Execution Planner
- Concepts: Graphs, sets, algorithms
- Problem: Given tasks and prerequisite pairs (A,B) meaning A must finish before B, return a valid execution order.
- Requirements: Detect cycles and raise ValueError with a clear message. For multiple available tasks, select alphabetically first. Reject unknown task names.
- Sample / expected behavior: Tasks=["build","test","deploy"]; dependencies=[("build","test"),("test","deploy")] -> ["build","test","deploy"].


In [ ]:
import heapq


def plan_execution(tasks, dependencies):
    """Return a valid execution order, choosing the alphabetically first
    ready task at every step.

    Raises ValueError for duplicate tasks, unknown task names, or cycles.
    """
    task_set = set(tasks)
    if len(task_set) != len(tasks):
        raise ValueError("Duplicate task names in the task list")

    successors = {t: set() for t in tasks}   # task -> tasks that must come after it
    indegree = {t: 0 for t in tasks}         # task -> number of unfinished prerequisites

    # ---- Build the graph ----
    for before, after in dependencies:
        for name in (before, after):
            if name not in task_set:
                raise ValueError(
                    f"Unknown task {name!r} in dependency ({before!r}, {after!r})"
                )
        if after not in successors[before]:      # ignore repeated pairs
            successors[before].add(after)
            indegree[after] += 1

    # ---- Kahn's algorithm with a min-heap ----
    ready = [t for t in tasks if indegree[t] == 0]
    heapq.heapify(ready)
    order = []

    while ready:
        current = heapq.heappop(ready)           # alphabetically first ready task
        order.append(current)
        for nxt in successors[current]:
            indegree[nxt] -= 1
            if indegree[nxt] == 0:
                heapq.heappush(ready, nxt)

    # ---- Cycle check ----
    if len(order) != len(tasks):
        stuck = sorted(t for t in tasks if indegree[t] > 0)
        raise ValueError(f"Cycle detected: these tasks can never run: {stuck}")

    return order

In [ ]:
print(plan_execution(["build", "test", "deploy"],
                     [("build", "test"), ("test", "deploy")]))
# ['build', 'test', 'deploy']

['build', 'test', 'deploy']


Task 05 | Expression Validator and Evaluator
- Concepts: Stacks, parsing, exception handling
- Problem: Create a calculator for integer expressions containing +, -, *, /, parentheses, and whitespace. Apply standard operator precedence and parentheses.
- Requirements: Do not use eval(), exec(), or third-party parsers. Handle unary minus, malformed expressions, and division by zero.
Sample / expected behavior: Input: "-(3 + 2) * 4 + 18 / 3" -> -14.0.


In [ ]:
DIGITS = "0123456789"


def tokenize(text):
    """Turn text into a list of (kind, value, position) tuples."""
    tokens = []
    i = 0
    n = len(text)
    while i < n:
        ch = text[i]
        if ch.isspace():
            i += 1
        elif ch in DIGITS:
            start = i
            while i < n and text[i] in DIGITS:
                i += 1
            tokens.append(("num", int(text[start:i]), start))
        elif ch in "+-*/()":
            tokens.append(("op", ch, i))
            i += 1
        else:
            raise ValueError(f"Unexpected character {ch!r} at position {i}")
    return tokens


def divide(a, b):
    """Integer division that truncates toward zero (-7 / 2 -> -3)."""
    if b == 0:
        raise ZeroDivisionError("Division by zero")
    q = abs(a) // abs(b)
    return q if (a < 0) == (b < 0) else -q


class Parser:
    def __init__(self, tokens):
        self.tokens = tokens
        self.pos = 0                      # index of the next unread token

    def peek(self):
        """Look at the next token without consuming it (None at the end)."""
        return self.tokens[self.pos] if self.pos < len(self.tokens) else None

    def next_is_op(self, *ops):
        tok = self.peek()
        return tok is not None and tok[0] == "op" and tok[1] in ops

    def parse(self):
        if not self.tokens:
            raise ValueError("Empty expression")
        value = self.expression()
        leftover = self.peek()
        if leftover is not None:
            raise ValueError(f"Unexpected {leftover[1]!r} at position {leftover[2]}")
        return value

    def expression(self):                 # handles + and -
        value = self.term()
        while self.next_is_op("+", "-"):
            op = self.tokens[self.pos][1]
            self.pos += 1
            right = self.term()
            value = value + right if op == "+" else value - right
        return value

    def term(self):                       # handles * and /
        value = self.factor()
        while self.next_is_op("*", "/"):
            op = self.tokens[self.pos][1]
            self.pos += 1
            right = self.factor()
            value = value * right if op == "*" else divide(value, right)
        return value

    def factor(self):                     # number, unary minus, or (...)
        tok = self.peek()
        if tok is None:
            raise ValueError("Unexpected end of expression")
        kind, val, pos = tok

        if kind == "num":
            self.pos += 1
            return val
        if val == "-":                    # unary minus
            self.pos += 1
            return -self.factor()
        if val == "(":
            self.pos += 1
            value = self.expression()
            closing = self.peek()
            if closing is None or closing[1] != ")":
                raise ValueError(f"Missing closing parenthesis for '(' at position {pos}")
            self.pos += 1
            return value
        raise ValueError(f"Unexpected {val!r} at position {pos}")


def calculate(text):
    """Evaluate an integer expression without eval()."""
    return Parser(tokenize(text)).parse()

In [ ]:
valid = [
    ("2 + 3 * (4 - 1)", 11),
    ("10 - 4 - 3", 3),          # left-associative
    ("8 / 2 / 2", 2),           # left-associative
    ("10 - 2 * 3", 4),          # precedence
    ("(2 + 3) * 4", 20),
    ("-2 * (3 + 4)", -14),
    ("2 * -3", -6),
    ("--3", 3),
    ("-(2 + 3)", -5),
    ("7 / 2", 3),
    ("-7 / 2", -3),             # truncates toward zero
    ("  42  ", 42),
    ("((2))", 2),
    ("1+2*3-4/2", 5),
]
for expr, expected in valid:
    assert calculate(expr) == expected, (expr, calculate(expr), expected)
print("All valid cases pass")

invalid = [
    "",              # Empty expression
    "   ",           # Empty expression
    "2 +",           # Unexpected end of expression
    "2 3",           # Unexpected 3 at position 2
    "(2 + 3",        # Missing closing parenthesis for '(' at position 0
    "2 + 3)",        # Unexpected ')' at position 5
    "2 * / 3",       # Unexpected '/' at position 4
    "()",            # Unexpected ')' at position 1
    "2 $ 3",         # Unexpected character '$' at position 2
    "3.5 + 1",       # Unexpected character '.' at position 1
    "5 / 0",         # ZeroDivisionError: Division by zero
    "5 / (3 - 3)",   # ZeroDivisionError: Division by zero
]
for expr in invalid:
    try:
        calculate(expr)
        print("NO ERROR (bug):", repr(expr))
    except (ValueError, ZeroDivisionError) as e:
        print(f"{expr!r:15} -> {type(e).__name__}: {e}")

All valid cases pass
''              -> ValueError: Empty expression
'   '           -> ValueError: Empty expression
'2 +'           -> ValueError: Unexpected end of expression
'2 3'           -> ValueError: Unexpected 3 at position 2
'(2 + 3'        -> ValueError: Missing closing parenthesis for '(' at position 0
'2 + 3)'        -> ValueError: Unexpected ')' at position 5
'2 * / 3'       -> ValueError: Unexpected '/' at position 4
'()'            -> ValueError: Unexpected ')' at position 1
'2 $ 3'         -> ValueError: Unexpected character '$' at position 2
'3.5 + 1'       -> ValueError: Unexpected character '.' at position 1
'5 / 0'         -> ZeroDivisionError: Division by zero
'5 / (3 - 3)'   -> ZeroDivisionError: Division by zero


Task 06 | Banking Ledger with OOP
Concepts: Classes, encapsulation, properties
Problem: Build Account and Bank classes. Support deposit, withdrawal, transfer, and account statements. Maintain immutable transaction records containing transaction ID, action, amount, and resulting balance.
Requirements: Reject nonpositive amounts, insufficient funds, and nonexistent accounts. A transfer must be atomic: both balances update or neither does.
Sample / expected behavior: Account A=1000, B=200; transfer 350 from A to B -> A=650, B=550.


In [ ]:
from dataclasses import dataclass
from itertools import count


# ---------- Errors ----------
class LedgerError(Exception):
    """Base class for every ledger problem."""


class InvalidAmountError(LedgerError):
    """Amount is not a positive whole number."""


class InsufficientFundsError(LedgerError):
    """Account does not have enough money."""


class AccountNotFoundError(LedgerError):
    """Account id does not exist."""


# ---------- Immutable record ----------
@dataclass(frozen=True)
class Transaction:
    txn_id: int
    action: str          # "open", "deposit", "withdraw", "transfer_out", "transfer_in"
    amount: int          # always positive; the action says the direction
    balance_after: int


# ---------- Account ----------
class Account:
    def __init__(self, account_id, opening_balance, next_txn_id):
        if not isinstance(account_id, str) or not account_id.strip():
            raise LedgerError("Account id must be a non-empty string")
        self._require_whole_number(opening_balance)
        if opening_balance < 0:
            raise InvalidAmountError("Opening balance cannot be negative")

        self.__account_id = account_id
        self.__balance = 0
        self.__history = []
        self.__next_txn_id = next_txn_id          # a function that returns a fresh ID
        self._apply("open", opening_balance, self.__next_txn_id())

    # ----- read-only properties -----
    @property
    def account_id(self):
        return self.__account_id

    @property
    def balance(self):
        return self.__balance

    # ----- validation helpers -----
    @staticmethod
    def _require_whole_number(value):
        if isinstance(value, bool) or not isinstance(value, int):
            raise InvalidAmountError(f"Amount must be a whole number, got {value!r}")

    @staticmethod
    def _validate_amount(amount):
        Account._require_whole_number(amount)
        if amount <= 0:
            raise InvalidAmountError(f"Amount must be positive, got {amount}")

    def _check_can_withdraw(self, amount):
        """Raise if this account cannot pay `amount`. Changes nothing."""
        self._validate_amount(amount)
        if amount > self.__balance:
            raise InsufficientFundsError(
                f"Account {self.__account_id!r} has {self.__balance}, "
                f"cannot withdraw {amount}"
            )

    def _apply(self, action, delta, txn_id):
        """The ONLY place the balance changes. Assumes checks already passed."""
        self.__balance += delta
        record = Transaction(txn_id, action, abs(delta), self.__balance)
        self.__history.append(record)
        return record

    # ----- public actions -----
    def deposit(self, amount):
        self._validate_amount(amount)
        return self._apply("deposit", amount, self.__next_txn_id())

    def withdraw(self, amount):
        self._check_can_withdraw(amount)
        return self._apply("withdraw", -amount, self.__next_txn_id())

    def statement(self):
        """Return a read-only copy of the history."""
        return tuple(self.__history)

    def format_statement(self):
        lines = [
            f"Statement for {self.__account_id}",
            f"{'ID':<4}{'Action':<14}{'Amount':>8}{'Balance':>10}",
        ]
        for t in self.__history:
            lines.append(f"{t.txn_id:<4}{t.action:<14}{t.amount:>8}{t.balance_after:>10}")
        return "\n".join(lines)


# ---------- Bank ----------
class Bank:
    def __init__(self):
        self.__accounts = {}              # account_id -> Account
        self.__txn_ids = count(1)         # 1, 2, 3, ...

    def _next_txn_id(self):
        return next(self.__txn_ids)

    def _get(self, account_id):
        try:
            return self.__accounts[account_id]
        except KeyError:
            raise AccountNotFoundError(f"Account {account_id!r} does not exist") from None

    def open_account(self, account_id, opening_balance=0):
        if account_id in self.__accounts:
            raise LedgerError(f"Account {account_id!r} already exists")
        account = Account(account_id, opening_balance, self._next_txn_id)
        self.__accounts[account_id] = account
        return account

    def balance(self, account_id):
        return self._get(account_id).balance

    def deposit(self, account_id, amount):
        return self._get(account_id).deposit(amount)

    def withdraw(self, account_id, amount):
        return self._get(account_id).withdraw(amount)

    def transfer(self, from_id, to_id, amount):
        # Phase 1: validate EVERYTHING. Nothing is changed in this phase.
        source = self._get(from_id)
        target = self._get(to_id)
        if source is target:
            raise LedgerError("Cannot transfer to the same account")
        source._check_can_withdraw(amount)       # checks amount AND funds

        # Phase 2: commit. Nothing below can fail.
        txn_id = self._next_txn_id()             # one ID shared by both records
        out_record = source._apply("transfer_out", -amount, txn_id)
        in_record = target._apply("transfer_in", amount, txn_id)
        return out_record, in_record

    def statement(self, account_id):
        return self._get(account_id).statement()

    def format_statement(self, account_id):
        return self._get(account_id).format_statement()


# ---------- Demo ----------
if __name__ == "__main__":
    bank = Bank()
    bank.open_account("A", 1000)     # txn 1
    bank.open_account("B", 200)      # txn 2
    bank.transfer("A", "B", 350)     # txn 3

    print("A =", bank.balance("A"))
    print("B =", bank.balance("B"))
    print()
    print(bank.format_statement("A"))
    print()
    print(bank.format_statement("B"))

A = 650
B = 550

Statement for A
ID  Action          Amount   Balance
1   open              1000      1000
3   transfer_out       350       650

Statement for B
ID  Action          Amount   Balance
2   open               200       200
3   transfer_in        350       550


Task 07 | Employee Payroll Engine
- Concepts: Inheritance, polymorphism, abstract classes
- Problem: Define abstract Employee with calculate_pay(). Implement SalariedEmployee, HourlyEmployee, and CommissionEmployee with different pay formulas. Generate a unified payroll summary.
- Requirements: Hourly overtime above 40 hours pays 1.5x. Commission employees receive base + sales*rate. Prevent direct instantiation of Employee.
- Sample / expected behavior: Hourly rate=20, hours=45 -> 950; Commission base=500, sales=4000, rate=.10 -> 900.


In [ ]:
import math
from abc import ABC, abstractmethod


# ---------- Validation helper ----------
def _require_number(label, value, maximum=None):
    """Return value if it is a finite number >= 0 (and <= maximum if given)."""
    if isinstance(value, bool) or not isinstance(value, (int, float)):
        raise TypeError(f"{label} must be a number, got {value!r}")
    if not math.isfinite(value):
        raise ValueError(f"{label} must be a finite number, got {value}")
    if value < 0:
        raise ValueError(f"{label} cannot be negative, got {value}")
    if maximum is not None and value > maximum:
        raise ValueError(f"{label} cannot be more than {maximum}, got {value}")
    return value


# ---------- Parent (abstract) ----------
class Employee(ABC):
    role = "Employee"                      # label shown in the report

    def __init__(self, name):
        if not isinstance(name, str) or not name.strip():
            raise ValueError("Name must be a non-empty string")
        self.name = name.strip()

    @abstractmethod
    def calculate_pay(self):
        """Return this employee's pay for one pay period."""


# ---------- Children ----------
class SalariedEmployee(Employee):
    role = "Salaried"

    def __init__(self, name, salary):
        super().__init__(name)
        self.salary = _require_number("Salary", salary)

    def calculate_pay(self):
        return round(self.salary, 2)


class HourlyEmployee(Employee):
    role = "Hourly"
    OVERTIME_AFTER = 40                    # hours
    OVERTIME_MULTIPLIER = 1.5

    def __init__(self, name, hourly_rate, hours_worked):
        super().__init__(name)
        self.hourly_rate = _require_number("Hourly rate", hourly_rate)
        self.hours_worked = _require_number("Hours worked", hours_worked)

    def calculate_pay(self):
        regular_hours = min(self.hours_worked, self.OVERTIME_AFTER)
        overtime_hours = max(self.hours_worked - self.OVERTIME_AFTER, 0)
        regular_pay = regular_hours * self.hourly_rate
        overtime_pay = overtime_hours * self.hourly_rate * self.OVERTIME_MULTIPLIER
        return round(regular_pay + overtime_pay, 2)


class CommissionEmployee(Employee):
    role = "Commission"

    def __init__(self, name, base_salary, sales, commission_rate):
        super().__init__(name)
        self.base_salary = _require_number("Base salary", base_salary)
        self.sales = _require_number("Sales", sales)
        self.commission_rate = _require_number("Commission rate", commission_rate, maximum=1)

    def calculate_pay(self):
        return round(self.base_salary + self.sales * self.commission_rate, 2)


# ---------- Unified report (polymorphism) ----------
def payroll_summary(employees):
    lines = [f"{'Name':<10}{'Type':<12}{'Pay':>10}", "-" * 32]
    total = 0
    for emp in employees:
        if not isinstance(emp, Employee):
            raise TypeError(f"Expected an Employee, got {emp!r}")
        pay = emp.calculate_pay()          # each object uses ITS OWN formula
        total += pay
        lines.append(f"{emp.name:<10}{emp.role:<12}{pay:>10.2f}")
    lines.append("-" * 32)
    lines.append(f"{'TOTAL':<22}{round(total, 2):>10.2f}")
    return "\n".join(lines)


# ---------- Demo ----------
if __name__ == "__main__":
    staff = [
        SalariedEmployee("Ali", 3000),
        HourlyEmployee("Sara", 20, 45),
        CommissionEmployee("Omar", 500, 4000, 0.10),
    ]
    print(payroll_summary(staff))

Name      Type               Pay
--------------------------------
Ali       Salaried       3000.00
Sara      Hourly          950.00
Omar      Commission      900.00
--------------------------------
TOTAL                    4850.00


Task 08 | Cache Decorator with Expiration
- Concepts: Decorators, closures, functools, time
- Problem: Implement @ttl_cache(seconds) that caches results for repeated positional and keyword arguments and recomputes after expiration. Track cache hits and misses.
- Requirements: Preserve function metadata with functools.wraps. Handle equivalent keyword argument order; document behavior for unhashable inputs. Use time.monotonic().
- Sample / expected behavior: A call with x=2,y=3 followed by y=3,x=2 before expiry should return a cache hit.


In [ ]:
import inspect
import math
import time
from collections import namedtuple
from functools import wraps

CacheInfo = namedtuple("CacheInfo", ["hits", "misses", "size", "ttl"])


def ttl_cache(seconds):
    """Cache a function's results for `seconds` seconds.

    - Equivalent calls share one entry: f(x=2, y=3), f(y=3, x=2), f(2, 3),
      and f(2, y=3) are all the same call. Omitted defaults are filled in.
    - Unhashable arguments (list, dict, set) bypass the cache: the function
      runs normally, nothing is stored, and the call counts as a miss.
    - Exceptions are not cached.
    - Not thread-safe.
    """
    # ---- Validate the TTL ----
    if isinstance(seconds, bool) or not isinstance(seconds, (int, float)):
        raise TypeError(
            f"seconds must be a number, got {seconds!r}. "
            "Use @ttl_cache(10), with parentheses."
        )
    if not math.isfinite(seconds):
        raise ValueError(f"seconds must be finite, got {seconds}")
    if seconds < 0:
        raise ValueError(f"seconds cannot be negative, got {seconds}")

    def decorator(func):
        signature = inspect.signature(func)
        cache = {}                              # key -> (result, time_saved)
        stats = {"hits": 0, "misses": 0}

        def make_key(args, kwargs):
            bound = signature.bind(*args, **kwargs)   # TypeError if call is invalid
            bound.apply_defaults()                    # fill in omitted defaults
            parts = []
            for name, value in bound.arguments.items():
                kind = signature.parameters[name].kind
                if kind is inspect.Parameter.VAR_KEYWORD:     # the **kwargs bucket
                    value = tuple(sorted(value.items(), key=lambda item: item[0]))
                parts.append((name, value))
            return tuple(parts)

        @wraps(func)
        def wrapper(*args, **kwargs):
            key = make_key(args, kwargs)

            try:
                hash(key)
            except TypeError:                     # unhashable input: bypass the cache
                stats["misses"] += 1
                return func(*args, **kwargs)

            entry = cache.get(key)
            if entry is not None:
                result, saved_at = entry
                if time.monotonic() - saved_at < seconds:     # still fresh
                    stats["hits"] += 1
                    return result
                del cache[key]                                # expired: forget it

            stats["misses"] += 1
            result = func(*args, **kwargs)         # if this raises, nothing is stored
            cache[key] = (result, time.monotonic())
            return result

        def cache_info():
            now = time.monotonic()
            expired = [k for k, (_, saved_at) in cache.items()
                       if now - saved_at >= seconds]
            for k in expired:                      # drop stale entries so size is honest
                del cache[k]
            return CacheInfo(stats["hits"], stats["misses"], len(cache), seconds)

        def cache_clear():
            cache.clear()
            stats["hits"] = 0
            stats["misses"] = 0

        wrapper.cache_info = cache_info
        wrapper.cache_clear = cache_clear
        return wrapper

    return decorator


# ---------- Demo ----------
if __name__ == "__main__":
    @ttl_cache(0.5)
    def add(x, y=0):
        """Add two things."""
        print(f"  computing add({x!r}, {y!r})")
        return x + y

    print(add(x=2, y=3))           # miss
    print(add(y=3, x=2))           # hit: keyword order does not matter
    print(add(2, 3))               # hit: positional equals keyword
    print(add(2, y=3))             # hit: mixed style
    print(add.cache_info())

    time.sleep(0.6)                # let the entry expire
    print(add(2, 3))               # miss: recomputed
    print(add.cache_info())

    print(add.__name__, "|", add.__doc__)   # metadata preserved
    add.cache_clear()
    print(add.cache_info())

    print(add([1], [2]))           # unhashable: bypasses the cache
    print(add([1], [2]))           # runs again
    print(add.cache_info())

  computing add(2, 3)
5
5
5
5
CacheInfo(hits=3, misses=1, size=1, ttl=0.5)
  computing add(2, 3)
5
CacheInfo(hits=3, misses=2, size=1, ttl=0.5)
add | Add two things.
CacheInfo(hits=0, misses=0, size=0, ttl=0.5)
  computing add([1], [2])
[1, 2]
  computing add([1], [2])
[1, 2]
CacheInfo(hits=0, misses=2, size=0, ttl=0.5)


Task 09 | Streaming Log Inspector
- Concepts: Generators, iterators, file handling
- Problem: Write a generator read_errors(path) that streams a log file line by line, yielding (line_number, timestamp, message) only for ERROR records. Build summarize_errors() to count errors by date.
- Requirements: Do not load the whole file into memory. Skip malformed lines while tracking their count. Expected format: YYYY-MM-DD HH:MM:SS | LEVEL | message.
- Sample / expected behavior: Input line: 2026-11-07 10:30:00 | ERROR | Connection lost -> (1,"2026-11-07 10:30:00","Connection lost").


In [ ]:
from datetime import datetime

TIMESTAMP_FORMAT = "%Y-%m-%d %H:%M:%S"
TIMESTAMP_LENGTH = 19                       # len("2026-11-07 10:30:00")
KNOWN_LEVELS = {"DEBUG", "INFO", "WARNING", "ERROR", "CRITICAL"}


def parse_line(line):
    """Return (timestamp, level, message) or raise ValueError with a reason."""
    parts = line.split(" | ", 2)            # at most 3 pieces; message may contain ' | '
    if len(parts) != 3:
        raise ValueError("wrong number of fields")

    timestamp, level, message = (p.strip() for p in parts)

    # strptime also accepts things like "2026-1-5 1:2:3", so check the length too
    if len(timestamp) != TIMESTAMP_LENGTH:
        raise ValueError("invalid timestamp")
    try:
        datetime.strptime(timestamp, TIMESTAMP_FORMAT)   # rejects 2026-13-45, Feb 30...
    except ValueError:
        raise ValueError("invalid timestamp") from None

    if level not in KNOWN_LEVELS:
        raise ValueError("unknown level")
    if not message:
        raise ValueError("empty message")

    return timestamp, level, message


def read_errors(path, stats=None):
    """Stream a log file and yield (line_number, timestamp, message) for ERROR lines.

    `stats` (a dict) is filled in while reading:
      lines, errors, blank, malformed, malformed_reasons
    The counts are complete only after the generator is exhausted.
    """
    if stats is None:
        stats = {}
    for key in ("lines", "errors", "blank", "malformed"):
        stats.setdefault(key, 0)
    reasons = stats.setdefault("malformed_reasons", {})

    with open(path, "r", encoding="utf-8-sig", errors="replace") as f:
        for line_number, raw in enumerate(f, start=1):
            stats["lines"] += 1

            if not raw.strip():                       # blank line
                stats["blank"] += 1
                continue

            line = raw.rstrip("\r\n")                 # remove only the newline
            try:
                timestamp, level, message = parse_line(line)
            except ValueError as problem:
                stats["malformed"] += 1
                reason = str(problem)
                reasons[reason] = reasons.get(reason, 0) + 1
                continue                              # skip it, keep going

            if level == "ERROR":
                stats["errors"] += 1
                yield (line_number, timestamp, message)


def summarize_errors(path):
    """Return (errors_by_date, stats). errors_by_date is sorted by date."""
    stats = {}
    by_date = {}
    for _, timestamp, _ in read_errors(path, stats):
        date = timestamp[:10]                         # "YYYY-MM-DD"
        by_date[date] = by_date.get(date, 0) + 1
    return dict(sorted(by_date.items())), stats


# ---------- Demo ----------
if __name__ == "__main__":
    sample = (
        "2026-11-07 10:30:00 | ERROR | Connection lost\n"
        "2026-11-07 10:31:15 | INFO | Retrying connection\n"
        "\n"
        "2026-11-07 10:32:00 | ERROR | Timeout after 30s | host=db1\n"
        "2026-11-08 09:00:00 | ERROR | Disk almost full\n"
        "this line is garbage\n"
        "2026-13-45 99:99:99 | ERROR | Impossible date\n"
        "2026-11-08 09:05:00 | WARNING | Disk at 95%\n"
        "2026-11-08 09:10:00 | ERROR | \n"
        "2026-11-08 09:15:00 | FATAL | Unknown level\n"
        "2026-11-09 08:00:00 | ERROR | Service crashed\n"
    )
    with open("sample.log", "w", encoding="utf-8") as f:
        f.write(sample)

    print("--- Errors ---")
    for record in read_errors("sample.log"):
        print(record)

    by_date, stats = summarize_errors("sample.log")
    print("\n--- Errors by date ---")
    for date, n in by_date.items():
        print(date, n)

    print("\n--- Stats ---")
    print(stats)

--- Errors ---
(1, '2026-11-07 10:30:00', 'Connection lost')
(4, '2026-11-07 10:32:00', 'Timeout after 30s | host=db1')
(5, '2026-11-08 09:00:00', 'Disk almost full')
(11, '2026-11-09 08:00:00', 'Service crashed')

--- Errors by date ---
2026-11-07 2
2026-11-08 1
2026-11-09 1

--- Stats ---
{'lines': 11, 'errors': 4, 'blank': 1, 'malformed': 4, 'malformed_reasons': {'wrong number of fields': 1, 'invalid timestamp': 1, 'empty message': 1, 'unknown level': 1}}


Task 10 | CSV Inventory Reconciliation
- Concepts: CSV, file handling, validation
- Problem: Read inventory.csv (sku,name,quantity) and movements.csv (sku,type,units). Apply IN and OUT movements in order, then write updated_inventory.csv and rejected_movements.csv.
- Requirements: Reject unknown SKUs, negative or zero units, invalid movement types, and any OUT causing negative stock. Continue processing after rejected rows.
- Sample / expected behavior: SKU P01 starts at 10; OUT 4, OUT 9, IN 2 -> final 8; second movement rejected.


In [ ]:
import csv

VALID_TYPES = {"IN", "OUT"}


# ---------- Small helpers ----------
def clean(value):
    """Turn None (missing cell) into '' and strip spaces."""
    return (value or "").strip()


def require_columns(reader, needed, path):
    found = set(reader.fieldnames or [])
    missing = needed - found
    if missing:
        raise ValueError(f"{path} is missing column(s): {sorted(missing)}")


def parse_units(text):
    """Convert text to int, or raise ValueError('units must be a whole number')."""
    if "_" in text:                      # int("1_0") would wrongly give 10
        raise ValueError("units must be a whole number")
    try:
        return int(text)
    except ValueError:
        raise ValueError("units must be a whole number") from None


# ---------- Step 1: load the inventory ----------
def load_inventory(path):
    """Return {sku: {"name": str, "quantity": int}}. Stops on bad master data."""
    inventory = {}
    with open(path, newline="", encoding="utf-8-sig") as f:
        reader = csv.DictReader(f)
        require_columns(reader, {"sku", "name", "quantity"}, path)
        for row in reader:
            line = reader.line_num
            sku = clean(row["sku"])
            if not sku:
                raise ValueError(f"{path} line {line}: empty SKU")
            if sku in inventory:
                raise ValueError(f"{path} line {line}: duplicate SKU {sku!r}")
            try:
                quantity = int(clean(row["quantity"]))
            except ValueError:
                raise ValueError(f"{path} line {line}: quantity must be a whole number") from None
            if quantity < 0:
                raise ValueError(f"{path} line {line}: quantity cannot be negative")
            inventory[sku] = {"name": clean(row["name"]), "quantity": quantity}
    return inventory


# ---------- Step 2: validate ONE movement (changes nothing) ----------
def validate_movement(row, inventory):
    """Return (sku, movement_type, units) or raise ValueError(reason)."""
    sku = clean(row.get("sku"))
    movement_type = clean(row.get("type")).upper()
    units_text = clean(row.get("units"))

    if sku not in inventory:
        raise ValueError("unknown SKU")
    if movement_type not in VALID_TYPES:
        raise ValueError("invalid movement type")

    units = parse_units(units_text)
    if units <= 0:
        raise ValueError("units must be positive")

    in_stock = inventory[sku]["quantity"]
    if movement_type == "OUT" and units > in_stock:
        raise ValueError(f"insufficient stock: have {in_stock}, need {units}")

    return sku, movement_type, units


# ---------- Step 3: process all movements ----------
def process_movements(path, inventory):
    """Apply valid movements in order. Return (rejected_rows, total, applied)."""
    rejected = []
    total = applied = 0
    with open(path, newline="", encoding="utf-8-sig") as f:
        reader = csv.DictReader(f)
        require_columns(reader, {"sku", "type", "units"}, path)
        for row in reader:
            total += 1
            try:
                sku, movement_type, units = validate_movement(row, inventory)
            except ValueError as problem:
                rejected.append({
                    "line_number": reader.line_num,
                    "sku": row.get("sku"),
                    "type": row.get("type"),
                    "units": row.get("units"),
                    "reason": str(problem),
                })
                continue                                   # keep going

            change = units if movement_type == "IN" else -units
            inventory[sku]["quantity"] += change           # the ONLY place stock changes
            applied += 1
    return rejected, total, applied


# ---------- Step 4: write the outputs ----------
def write_inventory(path, inventory):
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["sku", "name", "quantity"])
        writer.writeheader()
        for sku, item in inventory.items():
            writer.writerow({"sku": sku, "name": item["name"], "quantity": item["quantity"]})


def write_rejected(path, rejected):
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(
            f, fieldnames=["line_number", "sku", "type", "units", "reason"]
        )
        writer.writeheader()
        writer.writerows(rejected)


# ---------- Main entry point ----------
def reconcile(inventory_path="inventory.csv",
              movements_path="movements.csv",
              updated_path="updated_inventory.csv",
              rejected_path="rejected_movements.csv"):
    inventory = load_inventory(inventory_path)
    rejected, total, applied = process_movements(movements_path, inventory)
    write_inventory(updated_path, inventory)
    write_rejected(rejected_path, rejected)
    return total, applied, len(rejected)


# ---------- Demo ----------
if __name__ == "__main__":
    with open("inventory.csv", "w", newline="", encoding="utf-8") as f:
        f.write("sku,name,quantity\nP01,Widget,10\nP02,Gadget,5\nP03,Gizmo,0\n")

    with open("movements.csv", "w", newline="", encoding="utf-8") as f:
        f.write(
            "sku,type,units\n"
            "P01,OUT,4\n"
            "P01,OUT,9\n"
            "P01,IN,2\n"
            "P99,IN,5\n"
            "P02,MOVE,3\n"
            "P02,IN,0\n"
            "P02,OUT,-2\n"
            "P02,IN,abc\n"
            "P03,IN,7\n"
            "P02,OUT,5\n"
        )

    total, applied, rejected = reconcile()
    print(f"Movements: {total} | applied: {applied} | rejected: {rejected}\n")

    for name in ("updated_inventory.csv", "rejected_movements.csv"):
        print(f"--- {name} ---")
        with open(name, newline="", encoding="utf-8") as f:
            print(f.read())

Movements: 10 | applied: 4 | rejected: 6

--- updated_inventory.csv ---
sku,name,quantity
P01,Widget,8
P02,Gadget,0
P03,Gizmo,7

--- rejected_movements.csv ---
line_number,sku,type,units,reason
3,P01,OUT,9,"insufficient stock: have 6, need 9"
5,P99,IN,5,unknown SKU
6,P02,MOVE,3,invalid movement type
7,P02,IN,0,units must be positive
8,P02,OUT,-2,units must be positive
9,P02,IN,abc,units must be a whole number



Task 11 | Multi-File Student Gradebook
- Concepts: File handling, dictionaries, exceptions
- Problem: Merge students.csv (student_id,name) and scores.csv (student_id,subject,marks). Calculate per-student average, grade, and rank; export report.csv.
- Requirements: Marks must be 0–100; reject unknown IDs and duplicate student-subject entries. Grades: A>=85, B>=70, C>=55, D>=40, otherwise F. Rank by average, then ID.
- Sample / expected behavior: Student 101 with marks 90,80,85 -> average 85.0, grade A.


In [ ]:
import csv
import math

GRADE_CUTOFFS = ((85, "A"), (70, "B"), (55, "C"), (40, "D"))   # checked top to bottom


# ---------- Small helpers ----------
def clean(value):
    """Turn None (missing cell) into '' and strip spaces."""
    return (value or "").strip()


def require_columns(reader, needed, path):
    found = set(reader.fieldnames or [])
    missing = needed - found
    if missing:
        raise ValueError(f"{path} is missing column(s): {sorted(missing)}")


def parse_marks(text):
    """Convert text to a finite float, or raise ValueError('marks must be a number')."""
    if text == "" or "_" in text:            # float("1_0") would wrongly give 10.0
        raise ValueError("marks must be a number")
    try:
        value = float(text)
    except ValueError:
        raise ValueError("marks must be a number") from None
    if not math.isfinite(value):             # rejects 'nan' and 'inf'
        raise ValueError("marks must be a number")
    return value


def grade_for(average):
    for cutoff, letter in GRADE_CUTOFFS:
        if average >= cutoff:
            return letter
    return "F"


def id_sort_key(student_id):
    """All-digit IDs sort as numbers (99 before 100); others sort as text."""
    if student_id.isascii() and student_id.isdigit():
        return (0, int(student_id), student_id)
    return (1, 0, student_id)


# ---------- Step 1: load students ----------
def load_students(path):
    """Return {student_id: name}. Stops on bad master data."""
    students = {}
    with open(path, newline="", encoding="utf-8-sig") as f:
        reader = csv.DictReader(f)
        require_columns(reader, {"student_id", "name"}, path)
        for row in reader:
            line = reader.line_num
            student_id = clean(row["student_id"])
            name = clean(row["name"])
            if not student_id:
                raise ValueError(f"{path} line {line}: empty student_id")
            if not name:
                raise ValueError(f"{path} line {line}: empty name")
            if student_id in students:
                raise ValueError(f"{path} line {line}: duplicate student_id {student_id!r}")
            students[student_id] = name
    return students


# ---------- Step 2: validate ONE score row (changes nothing) ----------
def validate_score(row, students, scores_by_student):
    """Return (student_id, subject, marks) or raise ValueError(reason)."""
    student_id = clean(row.get("student_id"))
    subject = clean(row.get("subject"))
    marks_text = clean(row.get("marks"))

    if student_id not in students:
        raise ValueError("unknown student ID")
    if not subject:
        raise ValueError("empty subject")

    marks = parse_marks(marks_text)
    if marks < 0 or marks > 100:
        raise ValueError("marks must be between 0 and 100")

    already = scores_by_student.get(student_id, {})
    if subject.casefold() in already:                       # dictionary membership
        raise ValueError("duplicate student-subject entry")

    return student_id, subject, marks


# ---------- Step 3: read all scores ----------
def load_scores(path, students):
    """Return (scores_by_student, rejected_rows, total_rows)."""
    scores_by_student = {}
    rejected = []
    total = 0
    with open(path, newline="", encoding="utf-8-sig") as f:
        reader = csv.DictReader(f)
        require_columns(reader, {"student_id", "subject", "marks"}, path)
        for row in reader:
            total += 1
            try:
                student_id, subject, marks = validate_score(row, students, scores_by_student)
            except ValueError as problem:
                rejected.append({
                    "line_number": reader.line_num,
                    "student_id": row.get("student_id"),
                    "subject": row.get("subject"),
                    "marks": row.get("marks"),
                    "reason": str(problem),
                })
                continue                                    # keep going

            # the ONLY place a score is stored
            scores_by_student.setdefault(student_id, {})[subject.casefold()] = marks
    return scores_by_student, rejected, total


# ---------- Step 4: averages, grades, ranks ----------
def build_report(students, scores_by_student):
    rows = []
    for student_id, subjects in scores_by_student.items():
        average = round(sum(subjects.values()) / len(subjects), 2)
        rows.append({
            "student_id": student_id,
            "name": students[student_id],
            "average": average,
            "grade": grade_for(average),
        })

    rows.sort(key=lambda r: (-r["average"], id_sort_key(r["student_id"])))

    for position, row in enumerate(rows, start=1):
        row["rank"] = position
    return rows


# ---------- Step 5: write outputs ----------
def write_report(path, rows):
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["student_id", "name", "average", "grade", "rank"])
        writer.writeheader()
        for row in rows:
            writer.writerow({**row, "average": f"{row['average']:.2f}"})


def write_rejected(path, rejected):
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(
            f, fieldnames=["line_number", "student_id", "subject", "marks", "reason"]
        )
        writer.writeheader()
        writer.writerows(rejected)


# ---------- Main entry point ----------
def run_gradebook(students_path="students.csv", scores_path="scores.csv",
                  report_path="report.csv", rejected_path="rejected_scores.csv"):
    students = load_students(students_path)
    scores_by_student, rejected, total = load_scores(scores_path, students)
    rows = build_report(students, scores_by_student)
    write_report(report_path, rows)
    write_rejected(rejected_path, rejected)
    return {
        "students": len(students),
        "reported": len(rows),
        "without_scores": len(students) - len(rows),
        "score_rows": total,
        "accepted": total - len(rejected),
        "rejected": len(rejected),
    }


# ---------- Demo ----------
if __name__ == "__main__":
    with open("students.csv", "w", newline="", encoding="utf-8") as f:
        f.write("student_id,name\n101,Ali\n102,Sara\n103,Omar\n104,Hina\n105,Zain\n")

    with open("scores.csv", "w", newline="", encoding="utf-8") as f:
        f.write(
            "student_id,subject,marks\n"
            "101,Math,90\n"
            "101,Science,80\n"
            "101,English,85\n"
            "102,Math,95\n"
            "102,Science,100\n"
            "103,Math,60\n"
            "103,Math,70\n"
            "999,Math,50\n"
            "104,Math,101\n"
            "104,Science,abc\n"
            "104,Math,0\n"
            "103,Science,50\n"
            "102,English,-5\n"
        )

    print(run_gradebook(), "\n")
    for name in ("report.csv", "rejected_scores.csv"):
        print(f"--- {name} ---")
        with open(name, newline="", encoding="utf-8") as f:
            print(f.read())

{'students': 5, 'reported': 4, 'without_scores': 1, 'score_rows': 13, 'accepted': 8, 'rejected': 5} 

--- report.csv ---
student_id,name,average,grade,rank
102,Sara,97.50,A,1
101,Ali,85.00,A,2
103,Omar,55.00,C,3
104,Hina,0.00,F,4

--- rejected_scores.csv ---
line_number,student_id,subject,marks,reason
8,103,Math,70,duplicate student-subject entry
9,999,Math,50,unknown student ID
10,104,Math,101,marks must be between 0 and 100
11,104,Science,abc,marks must be a number
14,102,English,-5,marks must be between 0 and 100



Task 12 | Custom Iterable Range Toolkit
- Concepts: Iterators, generators, dunder methods
- Problem: Create a ReversibleRange(start,stop,step) class supporting iteration, reversed iteration, len(), and membership tests without constructing a list.
- Requirements: Match Python range stop-exclusive semantics for positive and negative nonzero steps. Implement __iter__, __reversed__, __len__, and __contains__.
- Sample / expected behavior: ReversibleRange(2,11,3) -> [2,5,8]; reversed -> [8,5,2]; 5 in object -> True.


In [ ]:
class ReversibleRange:
    """A small version of range() that supports iteration, reversed(),
    len(), and `in`, without ever building a list."""

    def __init__(self, start, stop, step=1):
        for label, value in (("start", start), ("stop", stop), ("step", step)):
            if isinstance(value, bool) or not isinstance(value, int):
                raise TypeError(f"{label} must be an integer, got {value!r}")
        if step == 0:
            raise ValueError("step must not be zero")

        self._start = start
        self._stop = stop
        self._step = step

    # ---------- helper: how many items? ----------
    def _count(self):
        if self._step > 0:
            distance = self._stop - self._start
            size = self._step
        else:
            distance = self._start - self._stop
            size = -self._step

        if distance <= 0:
            return 0
        return (distance + size - 1) // size          # ceiling division

    # ---------- the four required dunder methods ----------
    def __iter__(self):
        index = 0
        count = self._count()
        while index < count:
            yield self._start + index * self._step
            index += 1

    def __reversed__(self):
        index = self._count() - 1
        while index >= 0:
            yield self._start + index * self._step
            index -= 1

    def __len__(self):
        return self._count()

    def __contains__(self, value):
        if isinstance(value, bool) or not isinstance(value, int):
            return False

        if self._step > 0:
            in_bounds = self._start <= value < self._stop
        else:
            in_bounds = self._stop < value <= self._start

        return in_bounds and (value - self._start) % self._step == 0

    # ---------- nice to have ----------
    def __repr__(self):
        return f"ReversibleRange({self._start}, {self._stop}, {self._step})"


# ---------- Demo ----------
if __name__ == "__main__":
    r = ReversibleRange(2, 11, 3)
    print(r)
    print("forward :", list(r))
    print("reversed:", list(reversed(r)))
    print("len     :", len(r))
    print("5 in r  :", 5 in r)
    print("6 in r  :", 6 in r)
    print("11 in r :", 11 in r)

    down = ReversibleRange(10, 0, -3)
    print("\n", down)
    print("forward :", list(down))
    print("reversed:", list(reversed(down)))
    print("len     :", len(down))
    print("4 in down:", 4 in down)

    print("\nempty   :", list(ReversibleRange(5, 2, 1)), len(ReversibleRange(5, 2, 1)))
    print("huge len:", len(ReversibleRange(0, 10**12, 7)))
    print("huge in :", 7_000_000 in ReversibleRange(0, 10**12, 7))

    # Self-check against Python's real range
    for start in range(-6, 7):
        for stop in range(-6, 7):
            for step in (-4, -3, -2, -1, 1, 2, 3, 5):
                mine, real = ReversibleRange(start, stop, step), range(start, stop, step)
                assert list(mine) == list(real)
                assert list(reversed(mine)) == list(reversed(real))
                assert len(mine) == len(real)
                assert all((v in mine) == (v in real) for v in range(-10, 11))
    print("\nAll comparisons with range() passed")

ReversibleRange(2, 11, 3)
forward : [2, 5, 8]
reversed: [8, 5, 2]
len     : 3
5 in r  : True
6 in r  : False
11 in r : False

 ReversibleRange(10, 0, -3)
forward : [10, 7, 4, 1]
reversed: [1, 4, 7, 10]
len     : 4
4 in down: True

empty   : [] 0
huge len: 142857142858
huge in : True

All comparisons with range() passed


Task 13 | Booking Conflict Scheduler
- Concepts: Datetime, sorting, interval algorithms
- Problem: Implement a room booking scheduler. Each booking has room, start datetime, end datetime, and booking ID. Accept bookings only when they do not overlap existing bookings for the same room.
- Requirements: Use datetime objects; reject end<=start and duplicate IDs. Adjacent bookings are allowed. Return conflicts for rejected requests.
- Sample / expected behavior: Room A: 09:00–10:00 and 10:00–11:00 accepted; 09:30–10:30 rejected.


In [ ]:
from bisect import bisect_left
from dataclasses import dataclass
from datetime import datetime


@dataclass(frozen=True)
class Booking:
    booking_id: str
    room: str
    start: datetime
    end: datetime


@dataclass(frozen=True)
class BookingResult:
    accepted: bool
    booking: Booking
    conflicts: tuple = ()           # existing bookings that block this request


def overlaps(start1, end1, start2, end2):
    """True if [start1, end1) and [start2, end2) share any time."""
    return start1 < end2 and start2 < end1


class BookingScheduler:
    def __init__(self):
        self._by_room = {}          # room -> list of Bookings sorted by start
        self._starts = {}           # room -> parallel list of start times (for bisect)
        self._ids = set()           # every accepted booking id

    # ---------- validation ----------
    @staticmethod
    def _validate(booking_id, room, start, end):
        if not isinstance(booking_id, str) or not booking_id.strip():
            raise ValueError("booking_id must be a non-empty string")
        if not isinstance(room, str) or not room.strip():
            raise ValueError("room must be a non-empty string")
        if not isinstance(start, datetime) or not isinstance(end, datetime):
            raise TypeError("start and end must be datetime objects")
        if (start.tzinfo is None) != (end.tzinfo is None):
            raise TypeError("start and end must both be naive or both timezone-aware")
        if end <= start:
            raise ValueError("end must be later than start")

    # ---------- main operation ----------
    def book(self, booking_id, room, start, end):
        booking_id = booking_id.strip() if isinstance(booking_id, str) else booking_id
        room = room.strip() if isinstance(room, str) else room
        self._validate(booking_id, room, start, end)

        if booking_id in self._ids:
            raise ValueError(f"duplicate booking id {booking_id!r}")

        new = Booking(booking_id, room, start, end)
        bookings = self._by_room.setdefault(room, [])
        starts = self._starts.setdefault(room, [])

        i = bisect_left(starts, start)          # where `new` would be inserted

        conflicts = []
        if i > 0 and overlaps(start, end, bookings[i - 1].start, bookings[i - 1].end):
            conflicts.append(bookings[i - 1])   # left neighbor reaches into us
        j = i
        while j < len(bookings) and bookings[j].start < end:
            conflicts.append(bookings[j])       # starts before we end
            j += 1

        if conflicts:
            return BookingResult(False, new, tuple(conflicts))   # nothing changed

        bookings.insert(i, new)
        starts.insert(i, start)
        self._ids.add(booking_id)
        return BookingResult(True, new)

    def bookings_for(self, room):
        """Return the room's bookings in time order (as a read-only tuple)."""
        return tuple(self._by_room.get(room.strip(), ()))


# ---------- Demo ----------
if __name__ == "__main__":
    def t(hhmm):
        return datetime(2026, 11, 9, int(hhmm[:2]), int(hhmm[3:]))

    scheduler = BookingScheduler()
    requests = [
        ("B1", "Room A", "09:00", "10:00"),
        ("B2", "Room A", "10:00", "11:00"),     # adjacent: allowed
        ("B3", "Room A", "09:30", "10:30"),     # overlaps B1 and B2
        ("B4", "Room B", "09:30", "10:30"),     # other room: allowed
        ("B5", "Room A", "11:00", "12:00"),     # adjacent to B2: allowed
        ("B6", "Room A", "08:00", "12:30"),     # swallows everything
    ]
    for booking_id, room, s, e in requests:
        r = scheduler.book(booking_id, room, t(s), t(e))
        if r.accepted:
            print(f"{booking_id} {room} {s}-{e}: ACCEPTED")
        else:
            ids = [c.booking_id for c in r.conflicts]
            print(f"{booking_id} {room} {s}-{e}: REJECTED, conflicts with {ids}")

    print("\nRoom A schedule:")
    for b in scheduler.bookings_for("Room A"):
        print(f"  {b.booking_id}: {b.start:%H:%M}-{b.end:%H:%M}")

    for label, call in [
        ("duplicate id", lambda: scheduler.book("B1", "Room C", t("09:00"), t("10:00"))),
        ("end == start", lambda: scheduler.book("B9", "Room C", t("09:00"), t("09:00"))),
        ("end < start",  lambda: scheduler.book("B9", "Room C", t("10:00"), t("09:00"))),
        ("text times",   lambda: scheduler.book("B9", "Room C", "09:00", "10:00")),
    ]:
        try:
            call()
        except (ValueError, TypeError) as err:
            print(f"{label}: {type(err).__name__}: {err}")

B1 Room A 09:00-10:00: ACCEPTED
B2 Room A 10:00-11:00: ACCEPTED
B3 Room A 09:30-10:30: REJECTED, conflicts with ['B1', 'B2']
B4 Room B 09:30-10:30: ACCEPTED
B5 Room A 11:00-12:00: ACCEPTED
B6 Room A 08:00-12:30: REJECTED, conflicts with ['B1', 'B2', 'B5']

Room A schedule:
  B1: 09:00-10:00
  B2: 10:00-11:00
  B5: 11:00-12:00
duplicate id: ValueError: duplicate booking id 'B1'
end == start: ValueError: end must be later than start
end < start: ValueError: end must be later than start
text times: TypeError: start and end must be datetime objects


Task 14 | Command-Line Contact Manager
- Concepts: Modules, JSON, CRUD, validation
- Problem: Build a menu-driven CLI contact manager with add, search, update, delete, list, and persistent JSON storage. Each contact has a unique ID, name, phone, and email.
- Requirements: Use separate functions/modules for storage and business logic. Handle missing/corrupt JSON safely without silently deleting existing data. Search names case-insensitively.
- Sample / expected behavior: Add ID=1, name="Ayesha"; restart application; search "aye" -> contact remains available.
